# MACE Cu interatomic potential

## What is MACE?

**MACE (Massively Atom-Centered Equivariant)** is a modern machine-learning interatomic potential designed to accurately predict **energies**, **forces**, and **atomic interactions** in materials with near-DFT accuracy but at a much lower computational cost.

It is based on **E(3)-equivariant neural networks**, meaning the model obeys the fundamental physical symmetries of atoms in 3D space:

- **Rotational equivariance** — rotating the structure rotates the forces.
- **Translational invariance** — shifting the whole structure does not change physical properties.
- **Permutation invariance** — swapping identical atoms does not change the prediction.

These symmetries are built directly into the MACE architecture, making it **physically faithful, stable for MD, and highly data-efficient**.

---

## How MACE Works ?

### **1. Local Atomic Environments**
Each atom “sees” only neighbors within a cutoff radius (e.g., 6 Å).  
The model uses these neighbors to build a **high-dimensional description** of the local environment.

### **2. Message Passing Neural Network**
Atoms exchange information using a **message-passing mechanism**:
1. Each atom sends messages to its neighbors.  
2. Messages are combined into features that describe atomic interactions.  
3. Multiple message-passing steps allow the model to capture higher-order many-body effects.

### **3. Equivariance to Rotations (E(3) Symmetry)**
MACE uses **spherical harmonics** and tensor algebra to ensure that:
- energies remain unchanged under rotation,
- forces transform correctly.


### **4. Learning Energies and Forces**
The model is trained on reference data (typically DFT):
- total energies  
- atomic forces  
- optionally stresses, charges, etc.  

Loss = *energy error* + *force error*, usually with higher weight on forces.

---

## Why MACE is so powerful?
- Achieves **near-DFT accuracy** with far fewer training structures.
- Stable and smooth **molecular dynamics** because symmetry is enforced by design.
- Works for molecules, solids, surfaces, and catalytic sites.
- Scales efficiently on GPUs.

---


## 1.1 Getting setup

**MACE** requires a modern version of **PyTorch** that includes **CUDA** support. **Google Colab** does not have a compatible PyTorch by default, so we must install it manually.

The following command installs the latest stable PyTorch build with CUDA 12.4, which is supported on Colab GPUs.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


Before training MACE, make sure the Colab runtime is using a GPU.
Colab does not enable **GPUs** automatically, so you must turn it on manually:

Go to the top menu: Runtime -> Change runtime type

Set Hardware accelerator -> GPU -> Save


In [ ]:
!pip install ase mace-torch


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
import mace
print(mace.__version__)


In [ ]:
# Import PyTorch
import torch

# Import the MACE calculator class
from mace.calculators.mace import MACECalculator

# Check whether CUDA (GPU support) is available in the current Colab runtime
print("CUDA available:", torch.cuda.is_available())

# Print the installed PyTorch version to confirm it matches the CUDA build
print("Torch version:", torch.__version__)


## 1.2 Generating a Small Training Dataset


In this step, we create a small dataset to train a **MACE** model. Instead of using expensive DFT calculations, we use **ASE’s** **EMT calculator** to quickly generate energies and forces. EMT only supports a few simple metals, so in this example we use fcc Cu bulk.

* We build a periodic Cu supercell.

* We run a short Langevin molecular dynamics simulation at 600 K to explore different atomic configurations.

* At regular intervals, we save snapshots of the structure and record their energies and forces.

* We split these snapshots into training and validation sets and store them in extended **XYZ** format, which is the dataset format used by MACE.

In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.io import write
from ase.md.langevin import Langevin
from ase import units
import numpy as np

# Start from a simple structure:
#    Use fcc copper (Cu) because EMT has parameters for Cu.
atoms = bulk('Cu', 'fcc', a=3.6).repeat((2, 2, 2))  # 32 atoms
atoms.calc = EMT()  # attach EMT calculator

# Set up a Langevin MD simulation to explore configurations
dyn = Langevin(
    atoms,
    timestep=1.0 * units.fs,   # 1 fs time step
    temperature_K=600,         # target temperature (K)
    friction=0.02              # damping/friction coefficient
)

frames = []

def collect(atoms=atoms):
    # Get energy and forces from the *original* atoms object (which has a calculator)
    energy = float(atoms.get_potential_energy())  # total energy in eV
    forces = atoms.get_forces()                   # forces in eV/Å

    # Make a copy of the current structure and attach labels
    a = atoms.copy()
    a.info['energy'] = energy
    a.arrays['forces'] = forces

    frames.append(a)

# Equilibrate a bit, then sample configurations every 5 steps
for step in range(300):
    dyn.run(1)              # advance MD by 1 step
    if step % 5 == 0:       # sample every 5 fs
        collect()

# Train/valid split and write extended-XYZ files for MACE
n = len(frames)
split = int(0.9 * n)

write("train.xyz", frames[:split], format="extxyz")
write("valid.xyz", frames[split:], format="extxyz")

print(f"Wrote {split} train and {n - split} valid frames.")


In [ ]:
!pip install ase

In [ ]:
from ase.io import read
from ase.visualize import view

frames = read("train.xyz", index=":5")
view(frames)


## 1.3 Training the MACE Interatomic Potential

Now that we have created `train.xyz` and `valid.xyz` containing fcc Cu bulk structures
(with energies and forces from the EMT calculator), we can train a **MACE**
neural-network interatomic potential.

The command below:

- reads the training and validation datasets,
- builds a MACE model with a cutoff radius of 6 Å,
- and optimizes it to reproduce the reference energies and forces from EMT.




## 1.4 MACE Training Parameters

The table below summarizes the main parameters :

| Parameter        | Typical Range / Values           | Tuning vs Default        | Comment |
|------------------|----------------------------------|--------------------------|---------|
| `--name`         | any string                       | Default / bookkeeping    | Name of the run; used for folder, logs, model filenames. Choose something descriptive. |
| `--train_file`   | path to `.xyz` or `.extxyz`      | Default / required       | Training dataset with energies and forces. Must be correctly formatted. |
| `--valid_file`   | path to `.xyz` or `.extxyz`      | Default / required       | Validation dataset used to monitor overfitting and stop training. |
| `--model`        | `MACE`, `MACEModel`, etc.        | Usually fixed            | Chooses the model type. For most cases, `MACE` is recommended. |
| `--r_max`        | **4.0–7.0 Å**                    | **Hyperparameter tuning** | Cutoff radius for local environments. Larger → more neighbors, more accurate but slower. 6.0 Å is a good default. |
| `--max_num_epochs` | ~20–200                        | Hyperparameter tuning    | Number of epochs (passes over training data). Too small → underfitting; too large → overfitting / wasted time. |
| `--batch_size`   | ~4–64                            | Hyperparameter tuning    | Number of configurations per gradient step. Larger batch → smoother gradients, more memory needed. |
| `--energy_weight`| ~0.001–1.0                       | Hyperparameter tuning    | Relative weight of energy error in the loss. Often set small when forces are prioritized. |
| `--forces_weight`| ~0.1–1.0                         | Hyperparameter tuning    | Relative weight of force error in the loss. Forces usually dominate learning, so this is often 1.0. |
| `--energy_key`   | matches data (e.g. `energy`)     | Default / data-specific  | Must match the key name where energies are stored in `atoms.info`. |
| `--forces_key`   | matches data (e.g. `forces`)     | Default / data-specific  | Must match the key name where forces are stored in `atoms.arrays`. |
| `--E0s`          | per-element values or `"isolated"` / `"average"` | Important physics / setup | Defines isolated-atom reference energies. Best practice is to encode them in the dataset and set `E0s="isolated"`. Here we use a simple manual value `{29: 0.0}` for Cu. |
| `--device`       | `cpu`, `cuda`, `mps`             | Environment-specific     | Selects computing device. `cuda` uses the GPU and is strongly recommended for training. |

For more advanced tuning, you would also explicitly set parameters like:

- `--num_channels` (model width, e.g. 64, 128, 256)  
- `--max_L` (symmetry order of messages, 0–2)  
- `--num_interactions` (number of message passing layers, usually 2)  
- `--correlation` (many-body order, usually 3)  
- `--max_ell` (angular resolution, usually 3)

In this tutorial we keep those at their robust default values and focus only on a
small set of clear, interpretable hyperparameters.



More info: https://mace-docs.readthedocs.io/en/latest/guide/training.html

In [ ]:
# List installed MACE command-line tools
!ls /usr/local/bin | grep mace


In [ ]:
!mace_run_train \
  --name Cu_demo \
  --train_file train.xyz \
  --valid_file valid.xyz \
  --model MACE \
  --r_max 6.0 \
  --max_num_epochs 50 \
  --batch_size 4 \
  --energy_weight 0.01 \
  --forces_weight 1.0 \
  --energy_key energy \
  --forces_key forces \
  --E0s '{29: 0.0}' \
  --device cuda \
  --seed 42


## 1.5 Training the MACE Model - Interpretation

In this step, we trained the  **MACE** on a small dataset of Cu configurations generated using the ASE **EMT** calculator.


**1. Verified the environment**
- Detected **CUDA** and used the GPU for training.  
- Confirmed the dataset contains only one element: **Cu (atomic number 29)**.

**2. Loaded the dataset**
- **Training set:** 54 configurations  
- **Validation set:** 6 configurations  
- Each configuration includes:
  - **Total energy** stored under the key `energy`
  - **Per-atom forces** stored under the key `forces`

**3. Built the MACE model**
The model architecture includes:
- **2 message-passing layers**
- **128 channels**
- **Body order up to 4** (captures many-body atomic interactions)
- **Cutoff radius:** 6.0 Å
- **Radial and angular basis** that encode geometric information

**4. Optimized the model**
- Optimizer: **Adam**
- Batch size: **4 structures**
- Learning rate: **0.01**
- Combined loss:
  - Energies weighted: 0.01  
  - Forces weighted: 1.0 (forces dominate during training)

Training ran for **50 epochs**, reporting validation errors after each one.

---

**Final Model Accuracy**

MACE reports model performance on both the training and validation sets:

| Dataset | RMSE Energy (meV/atom) | RMSE Forces (meV/Å) | Relative Force RMSE (%) |
|--------|--------------------------|----------------------|--------------------------|
| **Train** | 0.6 | 4.3 | 1.66% |
| **Valid** | 0.6 | 4.8 | 1.23% |


- **Energies:**  
  The model predicts energies within **1 meV/atom**, which is extremely accurate for this simplified example.

- **Forces:**  
  Validation force errors are around **4.8 meV/Å**, meaning the model closely reproduces the EMT forces.

- **Relative force error:**  
  Only **1.23%**, indicating excellent agreement with the reference data.


> **The trained MACE model successfully learned the EMT potential for fcc Cu and can now act as a fast surrogate model to predict energies and forces for similar Cu configurations.**

---

**Output Files**

After training, MACE saved:

- A raw model checkpoint:  
  `checkpoints/Cu_demo_run-XXX_epoch-49.pt`
- A simplified, compiled model for deployment:  
  `Cu_demo_compiled.model`
- Training logs and metadata inside the folder:  
  `Cu_demo/`

These outputs allow you to **reload the model** in ASE and perform energy and force predictions.

---



##1.6 Using the Trained MACE Model for Prediction

In this step, we load the compiled MACE model (`Cu_demo_compiled.model`) and use it
as an ASE calculator to predict the energy and forces for one configuration from
the validation set.

- `E_pred = atoms_val.get_potential_energy()`  
  returns the **total potential energy** of the structure in **eV**.

- `F_pred = atoms_val.get_forces()`  
  returns the **forces on each atom** as an `(N_atoms, 3)` array in **eV/Å**, where
  each row corresponds to the x, y, z components of the force on a single atom.

These predictions are made by the trained MACE potential, which has learned to
reproduce the reference EMT energies and forces. Given the low validation errors
we observed, the MACE
predictions should closely match those of EMT for similar Cu configurations.


In [ ]:
from ase.io import read
from mace.calculators.mace import MACECalculator

# Load the trained Cu MACE model
calc = MACECalculator(model_path="Cu_demo_compiled.model", device="cuda")  # use "cpu" if no GPU

# Take the first structure from the validation set
atoms_val = read("valid.xyz", index=0)

# Attach the MACE calculator to this structure
atoms_val.calc = calc

# Compute energy and forces with the trained MACE model
E_pred = atoms_val.get_potential_energy()  # total energy in eV
F_pred = atoms_val.get_forces()            # forces in eV/Å

print("Predicted energy (MACE):", E_pred)
print("First 3 force vectors (MACE):\n", F_pred[:3])


##1.7 Evaluating MACE Accuracy Against EMT



In this step, we compare the predictions on the validation set from our trained **MACE** model to the
reference energies and forces produced by the **EMT calculator** for the same Cu
structure. By computing the EMT energy and the
force vectors, we can quantify how closely the MACE potential reproduces the
original reference data. A low force MAE indicates that the trained model has
accurately learned the underlying potential.


In [ ]:
from ase.calculators.emt import EMT
import numpy as np

# Compute reference EMT energy and forces for the same Cu structure
a_ref = atoms_val.copy()
a_ref.calc = EMT()

E_ref = a_ref.get_potential_energy()
F_ref = a_ref.get_forces()

# Mean absolute error in energy
mae_energy = abs(E_pred - E_ref)

# Mean absolute error in force magnitude per atom
mae_force = np.mean(np.linalg.norm(F_pred - F_ref, axis=1))

print(f"Energy MAE vs. EMT (eV): {mae_energy:.2f}")
print(f"Force MAE vs. EMT (eV/Å): {mae_force:.2f}")



 The MAE of energy is **0.01 eV**, while The mae of forces is **0.01 eV/Å**, which is very low and indicates excellent agreement with the reference forces. Therefore, the trained MACE model has successfully learned to reproduce the EMT potential for Cu and can now serve as a fast and accurate surrogate for energy and force evaluations.


To visually assess how well the trained MACE potential reproduces the reference
EMT calculator, we compare their predictions on all structures in the validation set.

For each configuration in `valid.xyz`, we compute:

- the **total energy** using EMT and MACE,
- all **force components** (Fx, Fy, Fz) on every atom.

We then plot:

1. **EMT energy vs MACE energy**
   - Points close to the diagonal indicate excellent energy agreement.

2. **EMT force components vs MACE force components**  
   - Flattening all force components into a single 1D array allows us to see how
     individual force components compare between the two models.

We also compute the **root mean square error (RMSE)** for energies and forces
to quantify the overall agreement. A small RMSE confirms that the MACE model
has successfully learned the EMT potential in the region sampled by the data.


In [ ]:
from ase.io import read
from ase.calculators.emt import EMT
import numpy as np
import matplotlib.pyplot as plt

# Read all validation structures
valid_frames = read("valid.xyz", ":")

E_mace = []
E_emt = []
F_mace_list = []
F_emt_list = []

for at in valid_frames:
    # MACE prediction
    at_mace = at.copy()
    at_mace.calc = calc  # trained MACE calculator
    E_mace.append(at_mace.get_potential_energy())
    F_mace_list.append(at_mace.get_forces())

    # EMT reference
    at_emt = at.copy()
    at_emt.calc = EMT()
    E_emt.append(at_emt.get_potential_energy())
    F_emt_list.append(at_emt.get_forces())

E_mace = np.array(E_mace)
E_emt = np.array(E_emt)

# Flatten all force components into 1D arrays
F_mace = np.concatenate(F_mace_list).reshape(-1, 3)
F_emt = np.concatenate(F_emt_list).reshape(-1, 3)

F_mace_flat = F_mace.flatten()
F_emt_flat = F_emt.flatten()

# Compute RMSE for energies and forces
rmse_E = np.sqrt(np.mean((E_mace - E_emt)**2))
rmse_F = np.sqrt(np.mean((F_mace_flat - F_emt_flat)**2))

print(f"Energy RMSE (MACE vs EMT) [eV]: {rmse_E:.3f}")
print(f"Force RMSE (MACE vs EMT) [eV/Å]: {rmse_F:.3f}")

# ==== PLOTS ====
plt.figure(figsize=(10, 4))

# Energy scatter plot
plt.subplot(1, 2, 1)
plt.scatter(E_emt, E_mace, s=25)
e_min = min(E_emt.min(), E_mace.min())
e_max = max(E_emt.max(), E_mace.max())
plt.plot([e_min, e_max], [e_min, e_max], 'k--', linewidth=1)  # y = x line
plt.xlabel("EMT energy (eV)")
plt.ylabel("MACE energy (eV)")
plt.title("Energy: MACE vs EMT")

# Forces scatter plot (all components)
plt.subplot(1, 2, 2)
plt.scatter(F_emt_flat, F_mace_flat, s=5)
f_min = min(F_emt_flat.min(), F_mace_flat.min())
f_max = max(F_emt_flat.max(), F_mace_flat.max())
plt.plot([f_min, f_max], [f_min, f_max], 'k--', linewidth=1)  # y = x line
plt.xlabel("EMT force components (eV/Å)")
plt.ylabel("MACE force components (eV/Å)")
plt.title("Forces: MACE vs EMT")

plt.tight_layout()
plt.show()


##1.8 Running a Short Molecular Dynamics Simulation with MACE

In this step, we run a brief **molecular dynamics (MD) simulation** using the
trained MACE model as the force calculator. By assigning the MACE
potential to the Cu structure and using the **Velocity Verlet integrator**,
we simulate the atomic motion for 100 femtoseconds (100 × 1 fs timesteps).
This demonstrates how the trained interatomic potential can be used
directly in ASE for MD, geometry optimization, or other atomistic workflows.


In [ ]:
from ase.md.verlet import VelocityVerlet
from ase.io import Trajectory
from ase import units

# Copy the Cu structure and assign the trained MACE calculator
atoms_md = atoms_val.copy()
atoms_md.calc = calc  # MACE calculator

# Set up MD: Velocity Verlet with 1 fs timestep
dyn = VelocityVerlet(atoms_md, 1.0 * units.fs)

# Open a trajectory file to save the MD frames
traj = Trajectory("cu_mace_md.traj", "w", atoms_md)

# Attach a function that writes the current frame to the trajectory every step
dyn.attach(traj.write, interval=1)

# Run a short MD trajectory (10 × 10 = 100 steps total)
for i in range(10):
    dyn.run(10)

traj.close()
print("Saved frames to cu_mace_md.traj")


Plotting the potential energy during a molecular dynamics trajectory is a
standard diagnostic tool. It allows us to check whether the simulation is
numerically and physically stable. For an NVE integration such as Velocity
Verlet, the total energy should remain approximately constant, with only small
fluctuations. If we observe large oscillations or a systematic drift in the
energy, this indicates problems such as an unstable timestep, noisy forces, or
a poorly trained interatomic potential.

In [ ]:
from ase.io import Trajectory
import matplotlib.pyplot as plt

traj = Trajectory("cu_mace_md.traj")

energies = [atoms.get_potential_energy() for atoms in traj]
steps = list(range(len(energies)))

plt.plot(steps, energies, marker="o")
plt.xlabel("MD step")
plt.ylabel("Potential energy (eV)")
plt.title("MACE MD: Energy vs. Step")
plt.show()


The potential energy oscillates smoothly during the MD trajectory, forming
a clear sinusoidal pattern. This behavior is typical of vibrational motion
around a local minimum on the potential energy surface. The absence of energy
drift indicates that the Velocity Verlet
integration is stable and that the MACE potential provides consistent forces.


##1.9 Testing MACE on Clean and Defective Cu(111) Slabs

In this step, we test the transferability of the bulk-trained MACE model by applying it to surface environments. We construct a clean Cu(111) slab, create a surface vacancy, relax both structures with the MACE calculator, and compute an approximate surface vacancy formation energy to evaluate how the model behaves on configurations outside its training domain.


In [ ]:
from ase.build import fcc111
from ase.optimize import BFGS
import numpy as np

# 1) Build a clean Cu(111) slab
clean_slab = fcc111(
    'Cu',
    size=(3, 3, 4),   # 3x3 surface, 4 layers
    a=3.6,            # lattice parameter consistent with our bulk example
    vacuum=8.0        # vacuum in z-direction (Å)
)

print("Clean slab: ", len(clean_slab), "atoms")

# 2) Create a surface vacancy: remove one atom from the top layer
defect_slab = clean_slab.copy()

z = defect_slab.positions[:, 2]
z_max = z.max()

# Atoms in the topmost layer (within 0.5 Å of the max z)
top_layer_indices = np.where(z > z_max - 0.5)[0]

# Choose one "central" atom in the top layer to remove
i_vac = top_layer_indices[len(top_layer_indices) // 2]
print("Removing atom index", i_vac, "from the top layer to create a vacancy.")
del defect_slab[i_vac]

print("Defective slab: ", len(defect_slab), "atoms")

# 3) Relax both slabs with the trained MACE calculator
#    (calc must already be defined as your MACECalculator)

for label, atoms in [("clean", clean_slab), ("defect", defect_slab)]:
    atoms.calc = calc  # trained MACE model !!!
    dyn = BFGS(atoms, logfile=None)
    dyn.run(fmax=0.05)   # quick relaxation to fmax = 0.05 eV/Å
    print(f"{label.capitalize()} slab relaxed.")

# 4) Compute energies
E_clean = clean_slab.get_potential_energy()
N_clean = len(clean_slab)

E_defect = defect_slab.get_potential_energy()
N_defect = len(defect_slab)

# Approximate bulk chemical potential from the clean slab
mu_bulk = E_clean / N_clean

# Surface vacancy formation energy (toy EMT/MACE estimate):
# E_vac = E(defect) - (N_defect * mu_bulk)
E_vac = E_defect - N_defect * mu_bulk

print(f"Clean slab energy       : {E_clean:.3f} eV")
print(f"Defective slab energy   : {E_defect:.3f} eV")
print(f"Approx. vacancy energy  : {E_vac:.3f} eV")


In [ ]:
import nglview as nv

view_clean = nv.show_ase(clean_slab)
view_clean.add_unitcell()


view_clean.add_ball_and_stick(
    sphere_scale=1.0,
    radius_scale=0.08 )

view_clean


In [ ]:
import nglview as nv

view_clean = nv.show_ase(defect_slab)
view_clean.add_unitcell()


view_clean.add_ball_and_stick(
    sphere_scale=1.0,
    radius_scale=0.08
)

view_clean


## 1.10 Interpreting the Cu(111) Surface Vacancy Energy

From the geometry optimized clean and defective Cu(111) slabs, the approximate surface vacancy formation energy was found to be about 0.017 eV, i.e., very close to zero and slightly positive. In physical terms, a vacancy formation energy should be strictly positive, since removing an atom from a stable surface requires energy. A value this close to zero indicates that the predicted energetics are not reliable in this regime.

Such behavior is expected in the present case.

* The MACE model used here was trained only on a small set of bulk fcc Cu configurations, with no surface or defect environments included in the training data. As a result, the surface atoms in the slab lie well outside the model’s training domain, making the predictions for surface defects inherently unstable.

* EMT itself is a simple approximate potential that does not accurately capture surface energetics.

* The slab used in this study is relatively small (3×3×4, 36 atoms), which introduces significant finite-size effects, including strong interactions between periodic images and incomplete surface relaxation.

